<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 11: Filtreler Kenar

**YAPAY ZEKA MÜHENDİSLİĞİ** · Modül 11 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta11/hafta11_filtreler_kenar.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta11/hafta11_filtreler_kenar.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>

</div>

# Hafta 11 - Filtreler ve Kenar Algılama

Bu defterde OpenCV ile görüntü filtreleme ve kenar algılama tekniklerini öğreneceğiz:
- Bulanıklaştırma filtreleri (blur, Gaussian, median)
- Kenar algılama (Canny)
- Eşikleme (threshold, adaptive threshold)
- Morfolojik işlemler (dilate, erode, open, close)

## 1. Kütüphaneler ve Yardımcı Fonksiyonlar

### Gerekli Paketlerin Kurulumu

Aşağıdaki komut ile ihtiyaç duyulan Python paketlerini yüklüyoruz.

In [ ]:
!pip install opencv-python-headless -q

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `cv2` | Bilgisayarlı görü ve görüntü işleme (OpenCV) |
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def goster(basliklar, goruntular, cmap_listesi=None, satir=1):
    """Birden fazla görüntüyü yan yana gösterir."""
    n = len(goruntular)
    sutun = (n + satir - 1) // satir
    fig, axes = plt.subplots(satir, sutun, figsize=(5 * sutun, 5 * satir))
    if satir == 1 and sutun == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()
    for i in range(n):
        cmap = cmap_listesi[i] if cmap_listesi else ('gray' if len(goruntular[i].shape) == 2 else None)
        if len(goruntular[i].shape) == 3:
            axes[i].imshow(cv2.cvtColor(goruntular[i], cv2.COLOR_BGR2RGB))
        else:
            axes[i].imshow(goruntular[i], cmap=cmap)
        axes[i].set_title(basliklar[i], fontsize=11, fontweight='bold')
        axes[i].axis('off')
    for j in range(n, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

print(f"OpenCV sürümü: {cv2.__version__}")

## 2. Test Görüntüsü Oluşturma

Filtreleri test etmek için gürültülü (noisy) bir görüntü oluşturacağız.

In [ ]:
# Temel görüntü: şekiller içeren sahne
img_temiz = np.zeros((400, 500, 3), dtype=np.uint8)
img_temiz[:] = [240, 240, 240]  # Açık gri arka plan

# Şekiller ekle
cv2.rectangle(img_temiz, (30, 30), (150, 150), (50, 50, 200), -1)
cv2.circle(img_temiz, (300, 100), 70, (50, 200, 50), -1)
cv2.rectangle(img_temiz, (50, 220), (200, 370), (200, 100, 50), -1)
cv2.circle(img_temiz, (350, 300), 80, (100, 50, 200), -1)
cv2.line(img_temiz, (0, 200), (500, 200), (0, 0, 0), 2)

# Gürültü ekle (Gaussian noise)
gurultu = np.random.normal(0, 25, img_temiz.shape).astype(np.int16)
img_gurultulu = np.clip(img_temiz.astype(np.int16) + gurultu, 0, 255).astype(np.uint8)

# Tuz-biber gürültüsü
img_tuz_biber = img_temiz.copy()
n_tuz = int(0.02 * img_temiz.shape[0] * img_temiz.shape[1])
for _ in range(n_tuz):
    y, x = np.random.randint(0, img_temiz.shape[0]), np.random.randint(0, img_temiz.shape[1])
    img_tuz_biber[y, x] = [255, 255, 255]
for _ in range(n_tuz):
    y, x = np.random.randint(0, img_temiz.shape[0]), np.random.randint(0, img_temiz.shape[1])
    img_tuz_biber[y, x] = [0, 0, 0]

goster(
    ['Temiz Görüntü', 'Gaussian Gürültü', 'Tuz-Biber Gürültüsü'],
    [img_temiz, img_gurultulu, img_tuz_biber]
)

## 3. Bulanıklaştırma Filtreleri

| Filtre | Fonksiyon | Kullanım Alanı |
|--------|-----------|----------------|
| Ortalama (Box) | `cv2.blur()` | Genel bulanıklaştırma |
| Gaussian | `cv2.GaussianBlur()` | Doğal bulanıklaştırma |
| Medyan | `cv2.medianBlur()` | Tuz-biber gürültüsü giderme |

In [ ]:
# Farklı filtreler uygula
blur_ortalama = cv2.blur(img_gurultulu, (5, 5))
blur_gaussian = cv2.GaussianBlur(img_gurultulu, (5, 5), 0)
blur_medyan = cv2.medianBlur(img_gurultulu, 5)

goster(
    ['Gürültülü Orijinal', 'Ortalama Filtre (5x5)',
     'Gaussian Filtre (5x5)', 'Medyan Filtre (5)'],
    [img_gurultulu, blur_ortalama, blur_gaussian, blur_medyan],
    satir=2
)

### Filtre ve Kenar Tespiti

Görüntüye çeşitli filtreler uygulayarak gürültüyü azaltıyor ve kenarları tespit ediyoruz.

In [ ]:
# Farklı çekirdek (kernel) boyutları karşılaştırması
boyutlar = [3, 7, 11, 21]
basliklar = [f'Gaussian ({b}x{b})' for b in boyutlar]
sonuclar = [cv2.GaussianBlur(img_gurultulu, (b, b), 0) for b in boyutlar]

goster(basliklar, sonuclar, satir=1)

### Filtre ve Kenar Tespiti

Görüntüye çeşitli filtreler uygulayarak gürültüyü azaltıyor ve kenarları tespit ediyoruz.

In [ ]:
# Tuz-biber gürültüsü için en iyi filtre: Medyan
medyan_3 = cv2.medianBlur(img_tuz_biber, 3)
medyan_5 = cv2.medianBlur(img_tuz_biber, 5)
gaussian_tb = cv2.GaussianBlur(img_tuz_biber, (5, 5), 0)

goster(
    ['Tuz-Biber Orijinal', 'Medyan (3)', 'Medyan (5)', 'Gaussian (5x5)'],
    [img_tuz_biber, medyan_3, medyan_5, gaussian_tb],
    satir=1
)

print("Not: Medyan filtre, tuz-biber gürültüsünü Gaussian'dan çok daha iyi temizler!")

## 4. Kenar Algılama: Canny

Canny kenar algılama algoritması:
1. Gaussian bulanıklaştırma (gürültü azaltma)
2. Gradient hesaplama (Sobel)
3. Non-maximum suppression (ince kenarlar)
4. Çift eşikleme (hysteresis thresholding)

**Parametreler:** `cv2.Canny(image, threshold1, threshold2)`

In [ ]:
# Gri tonlamaya çevir
gray = cv2.cvtColor(img_temiz, cv2.COLOR_BGR2GRAY)

# Farklı eşik değerleri ile Canny
canny_50_100 = cv2.Canny(gray, 50, 100)
canny_100_200 = cv2.Canny(gray, 100, 200)
canny_150_250 = cv2.Canny(gray, 150, 250)
canny_30_80 = cv2.Canny(gray, 30, 80)

goster(
    ['Orijinal (Gri)', 'Canny (50, 100)',
     'Canny (100, 200)', 'Canny (150, 250)'],
    [gray, canny_50_100, canny_100_200, canny_150_250],
    cmap_listesi=['gray', 'gray', 'gray', 'gray'],
    satir=2
)

### Filtre ve Kenar Tespiti

Görüntüye çeşitli filtreler uygulayarak gürültüyü azaltıyor ve kenarları tespit ediyoruz.

In [ ]:
# Gürültülü görüntüde kenar algılama (önce bulanıklaştır)
gray_noisy = cv2.cvtColor(img_gurultulu, cv2.COLOR_BGR2GRAY)
gray_blurred = cv2.GaussianBlur(gray_noisy, (5, 5), 0)

canny_noisy = cv2.Canny(gray_noisy, 50, 150)
canny_blurred = cv2.Canny(gray_blurred, 50, 150)

goster(
    ['Gürültülü Görüntü', 'Canny (Gürültülü)',
     'Bulanıklaştırılmış', 'Canny (Bulanık Sonrası)'],
    [gray_noisy, canny_noisy, gray_blurred, canny_blurred],
    cmap_listesi=['gray', 'gray', 'gray', 'gray'],
    satir=2
)

print("İpucu: Kenar algılama öncesi bulanıklaştırma, gürültüden kaynaklanan sahte kenarları azaltır.")

## 5. Eşikleme (Thresholding)

Eşikleme, pikselleri belirli bir değere göre siyah veya beyaza dönüştürür.

| Yöntem | Açıklama |
|--------|----------|
| `cv2.THRESH_BINARY` | Eşik üstü → beyaz, altı → siyah |
| `cv2.THRESH_BINARY_INV` | Eşik üstü → siyah, altı → beyaz |
| `cv2.THRESH_OTSU` | Otomatik eşik belirleme |
| `cv2.adaptiveThreshold` | Bölgesel eşikleme |

In [ ]:
# Eşikleme örnekleri
gray_clean = cv2.cvtColor(img_temiz, cv2.COLOR_BGR2GRAY)

_, thresh_binary = cv2.threshold(gray_clean, 127, 255, cv2.THRESH_BINARY)
_, thresh_inv = cv2.threshold(gray_clean, 127, 255, cv2.THRESH_BINARY_INV)
_, thresh_otsu = cv2.threshold(gray_clean, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Adaptif eşikleme
adaptive_mean = cv2.adaptiveThreshold(
    gray_clean, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 11, 2
)
adaptive_gauss = cv2.adaptiveThreshold(
    gray_clean, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
)

goster(
    ['Orijinal (Gri)', 'Binary (127)', 'Binary Ters',
     'Otsu', 'Adaptif (Ortalama)', 'Adaptif (Gaussian)'],
    [gray_clean, thresh_binary, thresh_inv,
     thresh_otsu, adaptive_mean, adaptive_gauss],
    cmap_listesi=['gray'] * 6,
    satir=2
)

## 6. Morfolojik İşlemler

İkili (binary) görüntüler üzerinde yapılan şekil tabanlı işlemlerdir:

| İşlem | Fonksiyon | Etkisi |
|-------|-----------|--------|
| **Genişleme (Dilate)** | `cv2.dilate()` | Beyaz bölgeleri büyütür |
| **Aşındırma (Erode)** | `cv2.erode()` | Beyaz bölgeleri küçültür |
| **Açma (Open)** | `cv2.morphologyEx(MORPH_OPEN)` | Küçük gürültüleri temizler (erode → dilate) |
| **Kapama (Close)** | `cv2.morphologyEx(MORPH_CLOSE)` | Küçük delikleri kapatır (dilate → erode) |

In [ ]:
# İkili görüntü oluştur
img_morph = np.zeros((300, 400), dtype=np.uint8)
cv2.rectangle(img_morph, (50, 50), (180, 180), 255, -1)
cv2.circle(img_morph, (300, 150), 80, 255, -1)

# Gürültü ekle (küçük noktalar)
for _ in range(100):
    y, x = np.random.randint(0, 300), np.random.randint(0, 400)
    cv2.circle(img_morph, (x, y), 2, 255, -1)

# Yapısal eleman (kernel)
kernel = np.ones((5, 5), np.uint8)

# Morfolojik işlemler
dilated = cv2.dilate(img_morph, kernel, iterations=1)
eroded = cv2.erode(img_morph, kernel, iterations=1)
opened = cv2.morphologyEx(img_morph, cv2.MORPH_OPEN, kernel)
closed = cv2.morphologyEx(img_morph, cv2.MORPH_CLOSE, kernel)

goster(
    ['Orijinal', 'Genişleme (Dilate)', 'Aşındırma (Erode)',
     'Açma (Open)', 'Kapama (Close)'],
    [img_morph, dilated, eroded, opened, closed],
    cmap_listesi=['gray'] * 5,
    satir=2
)

### Görüntü İşleme

OpenCV fonksiyonları ile görüntü üzerinde çeşitli işlemler gerçekleştiriyoruz.

In [ ]:
# Morfolojik gradient ve top hat
gradient = cv2.morphologyEx(img_morph, cv2.MORPH_GRADIENT, kernel)
tophat = cv2.morphologyEx(img_morph, cv2.MORPH_TOPHAT, kernel)
blackhat = cv2.morphologyEx(img_morph, cv2.MORPH_BLACKHAT, kernel)

goster(
    ['Orijinal', 'Gradient (Dilate - Erode)', 'Top Hat (Img - Open)', 'Black Hat (Close - Img)'],
    [img_morph, gradient, tophat, blackhat],
    cmap_listesi=['gray'] * 4,
    satir=1
)

## 7. Uygulamalı Örnek: Fotoğrafta Kenar Algılama

### Filtre ve Kenar Tespiti

Görüntüye çeşitli filtreler uygulayarak gürültüyü azaltıyor ve kenarları tespit ediyoruz.

In [ ]:
# Gerçekçi bir sahne oluştur (bina benzeri)
sahne = np.ones((500, 600, 3), dtype=np.uint8) * 200  # Gri gökyüzü

# Zemin
sahne[350:500, :] = [80, 120, 80]

# Bina 1
cv2.rectangle(sahne, (50, 150), (200, 350), (100, 100, 140), -1)
# Pencereler
for py in range(170, 340, 45):
    for px in range(70, 190, 40):
        cv2.rectangle(sahne, (px, py), (px + 25, py + 30), (200, 220, 240), -1)

# Bina 2
cv2.rectangle(sahne, (250, 100), (420, 350), (120, 110, 100), -1)
# Pencereler
for py in range(120, 340, 50):
    for px in range(270, 410, 45):
        cv2.rectangle(sahne, (px, py), (px + 30, py + 35), (180, 200, 220), -1)

# Ağaç
cv2.rectangle(sahne, (480, 280), (500, 350), (40, 80, 40), -1)  # Gövde
cv2.circle(sahne, (490, 240), 50, (30, 130, 30), -1)  # Yapraklar

# Kenar algılama pipeline
gray_sahne = cv2.cvtColor(sahne, cv2.COLOR_BGR2GRAY)
blur_sahne = cv2.GaussianBlur(gray_sahne, (3, 3), 0)
edges_sahne = cv2.Canny(blur_sahne, 50, 150)

# Kenarları orijinal üzerine bindir
sahne_kenarli = sahne.copy()
sahne_kenarli[edges_sahne > 0] = [0, 255, 0]  # Kenarları yeşil yap

goster(
    ['Orijinal Sahne', 'Gri Tonlama',
     'Canny Kenarlar', 'Kenarlar Üst Üste'],
    [sahne, gray_sahne, edges_sahne, sahne_kenarli],
    cmap_listesi=[None, 'gray', 'gray', None],
    satir=2
)

### Filtre ve Kenar Tespiti

Görüntüye çeşitli filtreler uygulayarak gürültüyü azaltıyor ve kenarları tespit ediyoruz.

In [ ]:
# Tam işlem hattı: Gürültü giderme → Eşikleme → Morfoloji → Kenar algılama
pipeline_gri = cv2.cvtColor(img_gurultulu, cv2.COLOR_BGR2GRAY)
pipeline_blur = cv2.GaussianBlur(pipeline_gri, (5, 5), 0)
_, pipeline_thresh = cv2.threshold(pipeline_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
pipeline_morph = cv2.morphologyEx(pipeline_thresh, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
pipeline_edges = cv2.Canny(pipeline_blur, 50, 150)

goster(
    ['1. Gürültülü', '2. Bulanıklaştırma', '3. Otsu Eşikleme',
     '4. Morfolojik Kapama', '5. Canny Kenarlar'],
    [pipeline_gri, pipeline_blur, pipeline_thresh, pipeline_morph, pipeline_edges],
    cmap_listesi=['gray'] * 5,
    satir=2
)

print("Tam işlem hattı: Gürültü giderme → Eşikleme → Morfoloji → Kenar algılama")

## Özet

Bu defterde şunları öğrendik:
- **Bulanıklaştırma:** blur (ortalama), GaussianBlur, medianBlur
- **Kenar algılama:** Canny algoritması ve eşik değerlerinin etkisi
- **Eşikleme:** Binary, Otsu, Adaptive (Mean ve Gaussian)
- **Morfolojik işlemler:** Dilate, Erode, Open, Close, Gradient
- **İşlem hattı:** Gürültü giderme → eşikleme → morfoloji → kenar algılama

**Sonraki defter:** Çizim, metin ve kontur algılama!

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

© 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>